In [1]:
import numpy as np
import pandas as pd
from transformers import AutoImageProcessor, AutoModel
import torch
import os
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
print('device:',device)

device: cuda


In [4]:
if Path('/kaggle/input').exists():
    model_path = '/kaggle/input/models/metaresearch/dinov2/pytorch/base/1/'
    dataset_path = '/kaggle/input/datasets/xiaose/cityscapes/Cityspaces/images'
else:
    notebook_path = Path.cwd()
    root_path = notebook_path.parent.parent 
    cityscapes_dataset_path = root_path / 'data' / 'Cityspaces' / 'images'
    model_path = root_path / 'model'
    comma_dataset_path = root_path / 'data' / 'comma2k19'    
print(f'Comma path: {comma_dataset_path}')

Comma path: /home/hxastur/vscode-projects/autonomous-driving/data/comma2k19


In [9]:
chunks = [x.name for x in Path(comma_dataset_path).iterdir() if x.is_dir()]
chunks

['Chunk_1']

```
Dataset_chunk_n
|
+-- route_id (dongle_id|start_time)
    |
    +-- segment_number
        |
        +-- preview.png (first frame video)
        +-- raw_log.bz2 (raw capnp log, can be read with openpilot-tools: logreader)
        +-- video.hevc (video file, can be read with openpilot-tools: framereader)
        +-- processed_log/ (processed logs as numpy arrays, see format for details)
        +-- global_pos/ (global poses of camera as numpy arrays, see format for details)
```

In [84]:
def build_index():
    samples = []
    for chunk in chunks:
        chunk_path = comma_dataset_path / chunk
        routes = [x for x in Path(chunk_path).iterdir() if x.is_dir()]
        for route in routes: 
            segments = [x for x in Path(route).iterdir() if x.is_dir()]
            for segment in segments: 
                video_path = segment / 'video.hevc'
                
                steering_angle_path = segment / 'processed_log' / 'CAN' / 'steering_angle'
                steering_angle_value=np.load(steering_angle_path / 'value')
                steering_angle_t=np.load(steering_angle_path / 't')

                frame_times_path = segment / 'global_pose' / 'frame_times'
                frame_times = np.load(frame_times_path)

                steering_angle_idx = np.searchsorted(
                    steering_angle_t,
                    frame_times
                )
                for i,frame_time in enumerate(frame_times):
                    idx = steering_angle_idx[i]
                    if idx == 0:
                        nearest = 0
                    elif idx == len(steering_angle_t):
                        nearest = len(steering_angle_t) - 1
                    else:
                        left = idx - 1
                        right = idx
                    
                        if abs(frame_time - steering_angle_t[left]) < abs(frame_time - steering_angle_t[right]):
                            nearest = left
                        else:
                            nearest = right
                    sample = {
                    'video_path': video_path,
                    'chunk':chunk,
                    'route':route.name,
                    'segment':segment.name,
                    'frame_idx':i,
                    'frame_time':frame_time,
                    'steering_angle': steering_angle_value[nearest]
                    }
                    samples.append(sample)
    return samples
samples = build_index()
samples[1]

{'video_path': PosixPath('/home/hxastur/vscode-projects/autonomous-driving/data/comma2k19/Chunk_1/b0c9d2329ad1606b|2018-07-30--13-03-07/22/video.hevc'),
 'chunk': 'Chunk_1',
 'route': 'b0c9d2329ad1606b|2018-07-30--13-03-07',
 'segment': '22',
 'frame_idx': 1,
 'frame_time': np.float64(11215.100412),
 'steering_angle': np.float64(3.5)}

In [ ]:
# import subprocess
# out_frames_path = segment / 'frames'
# out_frames_path.mkdir(exist_ok=True)
# subrocess.run([
#     'ffmpeg',
#     '-i', video_path,
#     str(out_dir / "frame_%06d.jpg")], check=True)

In [ ]:
from torch.utils.data import Dataset

class CommaDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
        
    # def __len__(self):
    #     ...
    
    def __getitem__(self, idx):
        # video_path = 
        # frame = read_frame(video_path, frame_idx)
        # return frame, steering
        pass